## Pipeline de pré-processamento dos reviews do AirBnB, nas cidades de Lisboa e Rio de Janeiro, afim de pesquisa em Text mining pra uma tese de Mestrado em Ciência de dados.

- Universidade Européia
- Mestrado em Ciência de dados e Análise de Negócios
- Orientaora: Prof. Marina Cavique
- Autora: Gisele Maciel da Silva

### Fonte: https://insideairbnb.com/get-the-data/

Datasets:
- reviews.csv **(Rio de Janeiro)** snapshot: 26 September, 2025
- reviews.csv **(Lisboa)** snapshot: 21 September, 2025 

In [1]:
# ==========================================
# Importação das bibliotecas necessárias
# ==========================================
import pandas as pd
from langdetect import detect, LangDetectException # Detecta a língua do texto
import matplotlib.pyplot as plt
from collections import Counter
import re
import spacy
from spacy.lang.pt.stop_words import STOP_WORDS
import string
import seaborn as sns

In [2]:
# ==========================================
# mostra o texto completo em cada célula
# ==========================================
pd.set_option("display.max_colwidth", None)  

-----
## **Pré-processamento na língua portuguesa**
---------

-------------------
## **Importação do dataset dos reviews da plataforma AirBnB:<br><br>Cidade: Rio de Janeiro.**

------------------

In [3]:
# ==========================================
# Abrir o dataset do Rio de Janeiro na versão filtrada para português
# ==========================================
df_RJ_pt = pd.read_pickle("reviews_RJ_pt.pkl")
df_RJ_pt.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments,cidade,lang
17,17878,597736,2011-10-04,989464,Jose Mauro,"Muito Melhor do que ir para um hotel. Apt com excelente localização( 1 quarteirão da praia), muito organizado e limpo. O proprietário sempre muito solicito para resolver qualquer problema. O Apt também tem um terraço maravilhoso. Esta é a melhor maneira de conhecer o RIO\r<br/>\r<br/>",Rio de Janeiro,pt
18,17878,620065,2011-10-11,1157847,Gabriel,"Muito bom o apartamento, a pouco mais de uma quadra da praia e com muitos lugares para comer e sair por perto.",Rio de Janeiro,pt
34,248756,3425364,2013-01-28,4175433,Nelson,"O apartamento é maravilhoso, novo, bem decorado, muito bem equipado, possui tudo que se precisa. A localização é perfeita próxima a tudo, tanto de Copacabana quanto Ipanema. Quero agradecer a atenção do Bernardo durante a negociação e a simpatia e presteza da Vera como anfitriã. Toda minha família se sentiu acolhida e bem a vontade. Um abraço pra todos e com certeza recomendo este apartamento, como também me hospedarei novamente nele de volta ao Rio.",Rio de Janeiro,pt
36,248756,3620482,2013-02-26,4781492,Rui Nuno,"Excelente apartamento, com ótimas condições e muito bem localizado. Fica perto de tudo - praias, supermercado, padaria, metro e festa - e está muito bem decorado. Recomendamos claramente. Fomos fantasticamente recebidos e deram-nos boas dicas para visitar a cidade. Só ficámos duas noites, mas valeu bem.\r<br/>Obrigado por tudo e sejam felizes.",Rio de Janeiro,pt
40,248756,4272123,2013-04-23,4893381,Isabella,"Dias Maravilhosos! Ap. muito bem decorado e com ótima localização, tudo como combinado. Recomendo!",Rio de Janeiro,pt


In [4]:
# ==========================================
# Verificar valores nulos
# ==========================================
print("Valores nulos por coluna:\n", df_RJ_pt.isnull().sum())

Valores nulos por coluna:
 listing_id       0
id               0
date             0
reviewer_id      0
reviewer_name    2
comments         0
cidade           0
lang             0
dtype: int64


In [5]:
# ==========================================
# Conta quantas linhas duplicadas existem no dataset filtrado para português
# ==========================================
print("Número de linhas duplicadas:", df_RJ_pt.duplicated().sum())

Número de linhas duplicadas: 0


In [181]:
df_RJ_pt.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments,cidade,lang,char_count,word_count
17,17878,597736,2011-10-04,989464,Jose Mauro,"Muito Melhor do que ir para um hotel. Apt com excelente localização( 1 quarteirão da praia), muito organizado e limpo. O proprietário sempre muito solicito para resolver qualquer problema. O Apt também tem um terraço maravilhoso. Esta é a melhor maneira de conhecer o RIO\r<br/>\r<br/>",Rio de Janeiro,pt,283,47
18,17878,620065,2011-10-11,1157847,Gabriel,"Muito bom o apartamento, a pouco mais de uma quadra da praia e com muitos lugares para comer e sair por perto.",Rio de Janeiro,pt,111,22
34,248756,3425364,2013-01-28,4175433,Nelson,"O apartamento é maravilhoso, novo, bem decorado, muito bem equipado, possui tudo que se precisa. A localização é perfeita próxima a tudo, tanto de Copacabana quanto Ipanema. Quero agradecer a atenção do Bernardo durante a negociação e a simpatia e presteza da Vera como anfitriã. Toda minha família se sentiu acolhida e bem a vontade. Um abraço pra todos e com certeza recomendo este apartamento, como também me hospedarei novamente nele de volta ao Rio.",Rio de Janeiro,pt,454,75
36,248756,3620482,2013-02-26,4781492,Rui Nuno,"Excelente apartamento, com ótimas condições e muito bem localizado. Fica perto de tudo - praias, supermercado, padaria, metro e festa - e está muito bem decorado. Recomendamos claramente. Fomos fantasticamente recebidos e deram-nos boas dicas para visitar a cidade. Só ficámos duas noites, mas valeu bem.\r<br/>Obrigado por tudo e sejam felizes.",Rio de Janeiro,pt,345,52
40,248756,4272123,2013-04-23,4893381,Isabella,"Dias Maravilhosos! Ap. muito bem decorado e com ótima localização, tudo como combinado. Recomendo!",Rio de Janeiro,pt,98,14


-----
## **Função única de limpeza textual para remover quebras de linhas, tags HTML e números que atrapalham a semântica do texto.**
------

In [18]:
# ==========================================
# Função única de limpeza textual
# ==========================================
nlp = spacy.load("pt_core_news_lg")

def limpar_texto(texto):
    texto = str(texto)

    # Remover quebras de linha e tags HTML simples
    texto = texto.replace("\\r<br/>", " ")
    texto = texto.replace("<br/>", " ")
    
    # Remover tags HTML genéricas
    texto = re.sub(r"<.*?>", " ", texto)

    # Remover parênteses
    texto = texto.replace("(", " - ").replace(")", " - ")

    # Acrescentar espaço depois da pontuação, se não houver
    texto = re.sub(r"(?<=[.,;:!?])(?=\S)", " ", texto)

    # Remover números
    texto = re.sub(r"\d+", " ", texto)

    # Remover espaços múltiplos
    texto = re.sub(r"\s+", " ", texto)

    return texto.strip()

In [19]:
df_RJ_pt["texto_limpo"] = df_RJ_pt["comments"].apply(limpar_texto)
df_RJ_pt.head(20)

,listing_id,id,date,reviewer_id,reviewer_name,comments,cidade,lang,texto_limpo
17,17878,597736,2011-10-04,989464,Jose Mauro,"Muito Melhor do que ir para um hotel. Apt com excelente localização( 1 quarteirão da praia), muito organizado e limpo. O proprietário sempre muito solicito para resolver qualquer problema. O Apt também tem um terraço maravilhoso. Esta é a melhor maneira de conhecer o RIO\r<br/>\r<br/>",Rio de Janeiro,pt,"Muito Melhor do que ir para um hotel. Apt com excelente localização - quarteirão da praia - , muito organizado e limpo. O proprietário sempre muito solicito para resolver qualquer problema. O Apt também tem um terraço maravilhoso. Esta é a melhor maneira de conhecer o RIO"
18,17878,620065,2011-10-11,1157847,Gabriel,"Muito bom o apartamento, a pouco mais de uma quadra da praia e com muitos lugares para comer e sair por perto.",Rio de Janeiro,pt,"Muito bom o apartamento, a pouco mais de uma quadra da praia e com muitos lugares para comer e sair por perto."
34,248756,3425364,2013-01-28,4175433,Nelson,"O apartamento é maravilhoso, novo, bem decorado, muito bem equipado, possui tudo que se precisa. A localização é perfeita próxima a tudo, tanto de Copacabana quanto Ipanema. Quero agradecer a atenção do Bernardo durante a negociação e a simpatia e presteza da Vera como anfitriã. Toda minha família se sentiu acolhida e bem a vontade. Um abraço pra todos e com certeza recomendo este apartamento, como também me hospedarei novamente nele de volta ao Rio.",Rio de Janeiro,pt,"O apartamento é maravilhoso, novo, bem decorado, muito bem equipado, possui tudo que se precisa. A localização é perfeita próxima a tudo, tanto de Copacabana quanto Ipanema. Quero agradecer a atenção do Bernardo durante a negociação e a simpatia e presteza da Vera como anfitriã. Toda minha família se sentiu acolhida e bem a vontade. Um abraço pra todos e com certeza recomendo este apartamento, como também me hospedarei novamente nele de volta ao Rio."
36,248756,3620482,2013-02-26,4781492,Rui Nuno,"Excelente apartamento, com ótimas condições e muito bem localizado. Fica perto de tudo - praias, supermercado, padaria, metro e festa - e está muito bem decorado. Recomendamos claramente. Fomos fantasticamente recebidos e deram-nos boas dicas para visitar a cidade. Só ficámos duas noites, mas valeu bem.\r<br/>Obrigado por tudo e sejam felizes.",Rio de Janeiro,pt,"Excelente apartamento, com ótimas condições e muito bem localizado. Fica perto de tudo - praias, supermercado, padaria, metro e festa - e está muito bem decorado. Recomendamos claramente. Fomos fantasticamente recebidos e deram-nos boas dicas para visitar a cidade. Só ficámos duas noites, mas valeu bem. Obrigado por tudo e sejam felizes."
40,248756,4272123,2013-04-23,4893381,Isabella,"Dias Maravilhosos! Ap. muito bem decorado e com ótima localização, tudo como combinado. Recomendo!",Rio de Janeiro,pt,"Dias Maravilhosos! Ap. muito bem decorado e com ótima localização, tudo como combinado. Recomendo!"
41,248756,5662504,2013-07-10,6859166,Ligia,"Excelente localização, muito perto da praia e fácil acesso para outros pontos. Fomos muito bem recebidos, nossa anfitriã foi muito atenciosa na nossa chegada e durante a estadia. Tudo funcionou muito bem (TV, internet, etc) e nos sentimos em casa.\r<br/>Ponto de atenção: a rua é bastante movimentada, então pode haver um pouco de barulho vindo da rua à noite. Para quem não se incomoda, não atrapalha.\r<br/>",Rio de Janeiro,pt,"Excelente localização, muito perto da praia e fácil acesso para outros pontos. Fomos muito bem recebidos, nossa anfitriã foi muito atenciosa na nossa chegada e durante a estadia. Tudo funcionou muito bem - TV, internet, etc - e nos sentimos em casa. Ponto de atenção: a rua é bastante movimentada, então pode haver um pouco de barulho vindo da rua à noite. Para quem não se incomoda, não atrapalha."
42,248756,5789290,2013-07-16,5693647,Victor Hugo,"Apartamento muito bom, LIMPO e bem organizado. Fui bem recepcionado e tive um fim de sem

In [20]:
# ==========================================
# Salva o DataFrame pré-processado em um arquivo pickle, para uso posterior
# ==========================================    
df_RJ_pt.to_pickle("Reviews_RJ_texto_limpo.pkl")

-------------------
## **Importação do dataset dos reviews da plataforma AirBnB:<br><br>Cidade: Lisboa.**

------------------

In [31]:
# ==========================================
# Importar os datasets "reviews_Lisboa_pt.pkl" da cidade de Lisboa
# ==========================================
df_Lisboa_pt = pd.read_pickle("reviews_Lisboa_pt.pkl")
df_Lisboa_pt.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments,cidade,lang
0,1113142,60014456,2016-01-18,35314589,Andre,Adoramos o lugar. Muito bem localizado e prático. Fomos muito bem recepcionados pela Inês que nos prestou toda a assistência necessária.,Lisboa,pt
4,1113142,215013241,2017-11-27,154490966,Agenor,Apartamento muito bom. Anfitrião atencioso e solícito. Não tenho qualquer reclamação. Só elogios.,Lisboa,pt
6,1113142,577387176,2019-12-15,15145286,Nath,"O apartamento é bem localizado e tem os utensílios para uma estadia mais prolongada. No entanto, tive sérios problemas com o proprietário : mudaram a data do meu check in no dia que eu cheguei - eu viajei com toda minha família e tenho um bebe com menos de 2 anos. Há quedas correntes de energia. E a proprietária me cobrou de algo absurdo: o vidro do box que protege a área de banho se trincou sozinho. Poderia ter causado um enorme acidente ao meu filho e demais membros da família. E a proprietária insiste em me cobrar o valor como se eu ou alguém tivéssemos feito algo. Um técnico foi ao local e confirmou que isso acontecia com o tempo, fiz até um video do vidro se despedaçando. Quase nao dormi preocupada com meu filho e quando achei que merecia um pedido de ""desculpas"",. veio uma cobrança. Muito absurdo.",Lisboa,pt
12,6499,18879225,2014-09-02,17027029,Simone,"Ola Bruno,\r<br/>\r<br/>Tive um mes Fantástico em seu apartamento. O apartamento realmente está bem situado. Um elétrico passa a cada 15 min. em frente, bem como várias linhas de autocarro. Isto facilita para ir ao Centro de Lisboa. Pontos turísticos e restaurantes bem próximos.\r<br/>Apartamento aconchegante com facilidade em organizá-lo. Ótimo acesso a internet. Boa vizinhança.\r<br/>Sempre tive apoio do anfitriao Bruno, bem como otima comunicao online. Esteve sempre auxiliando e buscou resolver de imediato coisas inesperadas no apartamento.\r<br/>\r<br/>Obrigado! Até à próxima oportunidade de estar em Lisboa.",Lisboa,pt
13,6499,21074122,2014-10-11,7661611,Cláudio,"Encontramos o apartamento de Bruno exatamente conforme a descrição e as fotos, além de muito bem limpo, cuidado com muito carinho e equipado com tudo o que é necessário e prático. Lençóis e toalhas disponíveis, travesseiros e uma cama extremamente confortável. A localização é incrível, segura e perfeita para visitar Lisboa. Todos os monumentos e locais históricos de Belém estão a alguns minutos da caminhada mais agradável da cidade, em meio a jardins, praças e restaurantes, e muitas opções de transportes para as outras zonas estão à porta do prédio. Numa estada mais extensa, como a nossa, é importante que o anfitrião esteja sempre ao alcance, e a atenção e a disponibilidade de Bruno realmente impressionam. Nenhuma de nossas comunicações e nenhum pequeno ajuste ou providência que foi necessária demorou mais do que alguns minutos. Uma de nossas melhores experiências no AirBnb.",Lisboa,pt


In [32]:
# ==========================================
# Verificar valores nulos
# ==========================================
print("Valores nulos por coluna:\n", df_Lisboa_pt.isnull().sum())

Valores nulos por coluna:
 listing_id       0
id               0
date             0
reviewer_id      0
reviewer_name    3
comments         0
cidade           0
lang             0
dtype: int64


In [33]:
# ==========================================
# Conta quantas linhas duplicadas existem no dataset filtrado para português
# ==========================================
print("Número de linhas duplicadas:", df_Lisboa_pt.duplicated().sum())

Número de linhas duplicadas: 0


-----
## **Função única de limpeza textual para remover quebras de linhas, tags HTML, números, pontuação e caracteres especiais que atrapalham a semântica do texto.**
------

In [29]:
# ==========================================
# Função única de limpeza textual
# ==========================================
nlp = spacy.load("pt_core_news_lg")

def limpar_texto(texto):
    texto = str(texto)

    # Remover quebras de linha e tags HTML simples
    texto = texto.replace("\\r<br/>", " ")
    texto = texto.replace("<br/>", " ")
    
    # Remover tags HTML genéricas
    texto = re.sub(r"<.*?>", " ", texto)

    # Remover parênteses
    texto = texto.replace("(", " - ").replace(")", " - ")

    # Acrescentar espaço depois da pontuação, se não houver
    texto = re.sub(r"(?<=[.,;:!?])(?=\S)", " ", texto)

    # Remover números
    texto = re.sub(r"\d+", " ", texto)

    # Remover espaços múltiplos
    texto = re.sub(r"\s+", " ", texto)

    return texto.strip()

In [34]:
df_Lisboa_pt["texto_limpo"] = df_Lisboa_pt["comments"].apply(limpar_texto)
df_Lisboa_pt.head(20)

,listing_id,id,date,reviewer_id,reviewer_name,comments,cidade,lang,texto_limpo
0,1113142,60014456,2016-01-18,35314589,Andre,Adoramos o lugar. Muito bem localizado e prático. Fomos muito bem recepcionados pela Inês que nos prestou toda a assistência necessária.,Lisboa,pt,Adoramos o lugar. Muito bem localizado e prático. Fomos muito bem recepcionados pela Inês que nos prestou toda a assistência necessária.
4,1113142,215013241,2017-11-27,154490966,Agenor,Apartamento muito bom. Anfitrião atencioso e solícito. Não tenho qualquer reclamação. Só elogios.,Lisboa,pt,Apartamento muito bom. Anfitrião atencioso e solícito. Não tenho qualquer reclamação. Só elogios.
6,1113142,577387176,2019-12-15,15145286,Nath,"O apartamento é bem localizado e tem os utensílios para uma estadia mais prolongada. No entanto, tive sérios problemas com o proprietário : mudaram a data do meu check in no dia que eu cheguei - eu viajei com toda minha família e tenho um bebe com menos de 2 anos. Há quedas correntes de energia. E a proprietária me cobrou de algo absurdo: o vidro do box que protege a área de banho se trincou sozinho. Poderia ter causado um enorme acidente ao meu filho e demais membros da família. E a proprietária insiste em me cobrar o valor como se eu ou alguém tivéssemos feito algo. Um técnico foi ao local e confirmou que isso acontecia com o tempo, fiz até um video do vidro se despedaçando. Quase nao dormi preocupada com meu filho e quando achei que merecia um pedido de ""desculpas"",. veio uma cobrança. Muito absurdo.",Lisboa,pt,"O apartamento é bem localizado e tem os utensílios para uma estadia mais prolongada. No entanto, tive sérios problemas com o proprietário : mudaram a data do meu check in no dia que eu cheguei - eu viajei com toda minha família e tenho um bebe com menos de anos. Há quedas correntes de energia. E a proprietária me cobrou de algo absurdo: o vidro do box que protege a área de banho se trincou sozinho. Poderia ter causado um enorme acidente ao meu filho e demais membros da família. E a proprietária insiste em me cobrar o valor como se eu ou alguém tivéssemos feito algo. Um técnico foi ao local e confirmou que isso acontecia com o tempo, fiz até um video do vidro se despedaçando. Quase nao dormi preocupada com meu filho e quando achei que merecia um pedido de ""desculpas"", . veio uma cobrança. Muito absurdo."
12,6499,18879225,2014-09-02,17027029,Simone,"Ola Bruno,\r<br/>\r<br/>Tive um mes Fantástico em seu apartamento. O apartamento realmente está bem situado. Um elétrico passa a cada 15 min. em frente, bem como várias linhas de autocarro. Isto facilita para ir ao Centro de Lisboa. Pontos turísticos e restaurantes bem próximos.\r<br/>Apartamento aconchegante com facilidade em organizá-lo. Ótimo acesso a internet. Boa vizinhança.\r<br/>Sempre tive apoio do anfitriao Bruno, bem como otima comunicao online. Esteve sempre auxiliando e buscou resolver de imediato coisas inesperadas no apartamento.\r<br/>\r<br/>Obrigado! Até à próxima oportunidade de estar em Lisboa.",Lisboa,pt,"Ola Bruno, Tive um mes Fantástico em seu apartamento. O apartamento realmente está bem situado. Um elétrico passa a cada min. em frente, bem como várias linhas de autocarro. Isto facilita para ir ao Centro de Lisboa. Pontos turísticos e restaurantes bem próximos. Apartamento aconchegante com facilidade em organizá-lo. Ótimo acesso a internet. Boa vizinhança. Sempre tive apoio do anfitriao Bruno, bem como otima comunicao online. Esteve sempre auxiliando e buscou resolver de imediato coisas inesperadas no apartamento. Obrigado! Até à próxima oportunidade de estar em Lisboa."
13,6499,21074122,2014-10-11,7661611,Cláudio,"Encontramos o apartamento de Bruno exatamente conforme a descrição e as fotos, além de muito bem limpo, cuidado com muito carinho e equipado com tudo o que é necessário e prático. Lençóis e toalhas disponíveis, travesseiros e uma cama extremamente confortável. A localização é incrível, segura e perfeita para visitar Lisboa. Todos os monumento

In [35]:
# ==========================================
# Salvando o DataFrame pré-processado para Lisboa, e criando uma cópia de segurança
# ==========================================
df_Lisboa_pt.to_pickle("Reviews_Lisboa_texto_limpo.pkl")

---------------------
##  **Fazer a junção dos datasets, Rio de Janeiro e Lisboa**
------------

###  Balanceamento dos datasets em (50% / 50%), com base na coluna **'cidade'**, sendo:

- **Rio de Janeiro:** 100.000 entradas
- **Lisboa:** 100.000 entradas

------------

In [36]:
# ==========================================
# Carregar os datasets processados
# ==========================================
df_RJ_concat = pd.read_pickle("Reviews_RJ_texto_limpo.pkl")
df_Lisboa_concat = pd.read_pickle("Reviews_Lisboa_texto_limpo.pkl")

# ==========================================
# Verificar o tamanho dos datasets
# ==========================================
print("RJ:", len(df_RJ_concat))
print("Lisboa:", len(df_Lisboa_concat))

# ==========================================
# Definir o tamanho de balanceamento
# ==========================================
n_amostra = 100000

# ==========================================
# Subamostragem sem duplicar texto
# ==========================================
df_RJ_bal = df_RJ_concat.sample(n=n_amostra, random_state=42)
df_Lisboa_bal = df_Lisboa_concat.sample(n=n_amostra, random_state=42)

# ==========================================
# Concatenar os dois conjuntos balanceados
# ==========================================
df_balancear = pd.concat([df_RJ_bal, df_Lisboa_bal], ignore_index=True)



RJ: 721089
Lisboa: 124783


In [37]:
# ==========================================
# Contar entradas por cidade após o balanceamento de dados
# ==========================================
df_balancear['cidade'].value_counts()

cidade
Rio de Janeiro    100000
Lisboa            100000
Name: count, dtype: int64

In [38]:
# ==========================================
# Conferir dataset
# ==========================================
df_balancear.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments,cidade,lang,texto_limpo
0,1241990244433390890,1450109880518848792,2025-06-23,339707612,Iara,"Os anfitriões são maravilhosos, a casa é um charme com macaquinhos vindo toda manhã na varanda, porém considerando ter muitos móveis velhos e objetos pessoais acaba ficando com cheiro forte de coisa guardada e fechada",Rio de Janeiro,pt,"Os anfitriões são maravilhosos, a casa é um charme com macaquinhos vindo toda manhã na varanda, porém considerando ter muitos móveis velhos e objetos pessoais acaba ficando com cheiro forte de coisa guardada e fechada"
1,30938466,617993232260012422,2022-05-02,112589718,Jeison,"A estadia com o Renato foi a melhor experiência no Airbnb até hoje .<br/>Vou falar 4 coisas que não poderia deixar passar.<br/>01- A vista da cobertura é sensacional , um ângulo de visão da Bahia de Guanabara , Pão de Açucar , Aeroporto Santos Dumont e Cristo Redentor é sensacional.<br/>02- A recepção no Chek-in foi indescritível , fomos recebidos com algumas surpresas que me deixaram de queixo caído , uma delas é de uma simpatia sem igual, mas não posso contar a vocês porque perderia a graça.<br/>03- A flexibilidade na reserva e as dicas do RJ que o Renato passa , são maravilhosas.<br/>04- A localização do apartamento é fantástica pois fica pórximo a tudo.<br/>Enfim , Super indico o Renato para se hospedarem , o Apartamento é fantastico !!!",Rio de Janeiro,pt,"A estadia com o Renato foi a melhor experiência no Airbnb até hoje . Vou falar coisas que não poderia deixar passar. - A vista da cobertura é sensacional , um ângulo de visão da Bahia de Guanabara , Pão de Açucar , Aeroporto Santos Dumont e Cristo Redentor é sensacional. - A recepção no Chek-in foi indescritível , fomos recebidos com algumas surpresas que me deixaram de queixo caído , uma delas é de uma simpatia sem igual, mas não posso contar a vocês porque perderia a graça. - A flexibilidade na reserva e as dicas do RJ que o Renato passa , são maravilhosas. - A localização do apartamento é fantástica pois fica pórximo a tudo. Enfim , Super indico o Renato para se hospedarem , o Apartamento é fantastico ! ! !"
2,1279989569850103457,1490668744495176611,2025-08-18,12329668,Beatriz,"o apt do Maurício é otimo!!! tudo nele nos encantou. o prédio, os porteiros, o apt, a gentileza, a localização, enfim, um lugar com a cara do Rio e da gentileza e bom gosto do carioca",Rio de Janeiro,pt,"o apt do Maurício é otimo! ! ! tudo nele nos encantou. o prédio, os porteiros, o apt, a gentileza, a localização, enfim, um lugar com a cara do Rio e da gentileza e bom gosto do carioca"
3,1079929507092661295,1147883300514198966,2024-05-02,574751517,Gabriela,"muito organizado, bonito, limpo, funcional. anfitrião atencioso, os melhores horários de check-in e checkout que já vi. gelo, café, toalhas, roupa de cama tudo que precisei tinha. Supermercado e farmácia perto, excelente. a vila estava um pouco barulhenta a noite, mas entendo que isso está fora do controle do anfitrião. umica recomendação seria um insulfim espelhado na porta. mas pq eu sou mais sensível a claridade mesmo.",Rio de Janeiro,pt,"muito organizado, bonito, limpo, funcional. anfitrião atencioso, os melhores horários de check-in e checkout que já vi. gelo, café, toalhas, roupa de cama tudo que precisei tinha. Supermercado e farmácia perto, excelente. a vila estava um pouco barulhenta a noite, mas entendo que isso está fora do controle do anfitrião. umica recomendação seria um insulfim espelhado na porta. mas pq eu sou mais sensível a claridade mesmo."
4,572381918951779336,1247881696472392123,2024-09-17,172437167,Anna Paula,"Incrível! Amei tudo. Super rápida em soluções e respostas. Localização melhor, impossível!",Rio de Janeiro,pt,"Incrível! Amei tudo. Super rápida em soluções e respostas. Localização melhor, impossível!"


In [40]:
df_balancear.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   listing_id     200000 non-null  int64
 1   id             200000 non-null  int64
 2   date           200000 non-null  str  
 3   reviewer_id    200000 non-null  int64
 4   reviewer_name  199998 non-null  str  
 5   comments       200000 non-null  str  
 6   cidade         200000 non-null  str  
 7   lang           200000 non-null  str  
 8   texto_limpo    200000 non-null  str  
dtypes: int64(3), str(6)
memory usage: 54.8 MB


In [42]:
# ==========================================
# Remover várias colunas
# ==========================================
df_balancear = df_balancear.drop(columns=['id', 'listing_id', 'reviewer_id', 'reviewer_name'])
df_balancear.head()

,date,comments,cidade,lang,texto_limpo
0,2025-06-23,"Os anfitriões são maravilhosos, a casa é um charme com macaquinhos vindo toda manhã na varanda, porém considerando ter muitos móveis velhos e objetos pessoais acaba ficando com cheiro forte de coisa guardada e fechada",Rio de Janeiro,pt,"Os anfitriões são maravilhosos, a casa é um charme com macaquinhos vindo toda manhã na varanda, porém considerando ter muitos móveis velhos e objetos pessoais acaba ficando com cheiro forte de coisa guardada e fechada"
1,2022-05-02,"A estadia com o Renato foi a melhor experiência no Airbnb até hoje .<br/>Vou falar 4 coisas que não poderia deixar passar.<br/>01- A vista da cobertura é sensacional , um ângulo de visão da Bahia de Guanabara , Pão de Açucar , Aeroporto Santos Dumont e Cristo Redentor é sensacional.<br/>02- A recepção no Chek-in foi indescritível , fomos recebidos com algumas surpresas que me deixaram de queixo caído , uma delas é de uma simpatia sem igual, mas não posso contar a vocês porque perderia a graça.<br/>03- A flexibilidade na reserva e as dicas do RJ que o Renato passa , são maravilhosas.<br/>04- A localização do apartamento é fantástica pois fica pórximo a tudo.<br/>Enfim , Super indico o Renato para se hospedarem , o Apartamento é fantastico !!!",Rio de Janeiro,pt,"A estadia com o Renato foi a melhor experiência no Airbnb até hoje . Vou falar coisas que não poderia deixar passar. - A vista da cobertura é sensacional , um ângulo de visão da Bahia de Guanabara , Pão de Açucar , Aeroporto Santos Dumont e Cristo Redentor é sensacional. - A recepção no Chek-in foi indescritível , fomos recebidos com algumas surpresas que me deixaram de queixo caído , uma delas é de uma simpatia sem igual, mas não posso contar a vocês porque perderia a graça. - A flexibilidade na reserva e as dicas do RJ que o Renato passa , são maravilhosas. - A localização do apartamento é fantástica pois fica pórximo a tudo. Enfim , Super indico o Renato para se hospedarem , o Apartamento é fantastico ! ! !"
2,2025-08-18,"o apt do Maurício é otimo!!! tudo nele nos encantou. o prédio, os porteiros, o apt, a gentileza, a localização, enfim, um lugar com a cara do Rio e da gentileza e bom gosto do carioca",Rio de Janeiro,pt,"o apt do Maurício é otimo! ! ! tudo nele nos encantou. o prédio, os porteiros, o apt, a gentileza, a localização, enfim, um lugar com a cara do Rio e da gentileza e bom gosto do carioca"
3,2024-05-02,"muito organizado, bonito, limpo, funcional. anfitrião atencioso, os melhores horários de check-in e checkout que já vi. gelo, café, toalhas, roupa de cama tudo que precisei tinha. Supermercado e farmácia perto, excelente. a vila estava um pouco barulhenta a noite, mas entendo que isso está fora do controle do anfitrião. umica recomendação seria um insulfim espelhado na porta. mas pq eu sou mais sensível a claridade mesmo.",Rio de Janeiro,pt,"muito organizado, bonito, limpo, funcional. anfitrião atencioso, os melhores horários de check-in e checkout que já vi. gelo, café, toalhas, roupa de cama tudo que precisei tinha. Supermercado e farmácia perto, excelente. a vila estava um pouco barulhenta a noite, mas entendo que isso está fora do controle do anfitrião. umica recomendação seria um insulfim espelhado na porta. mas pq eu sou mais sensível a claridade mesmo."
4,2024-09-17,"Incrível! Amei tudo. Super rápida em soluções e respostas. Localização melhor, impossível!",Rio de Janeiro,pt,"Incrível! Amei tudo. Super rápida em soluções e respostas. Localização melhor, impossível!"


In [43]:
# ==========================================
# Salvar o dataset balanceado de Lisboa e Rio de Janeiro, para uso posterior
# ==========================================
df_balancear.to_pickle("Reviews_Lisboa_RJ_200k_texto_limpo.pkl")
print("Dataset balanceado com o texto limpo, salvo com sucesso!")

Dataset balanceado com o texto limpo, salvo com sucesso!


-------------
## Verifificando se os dois datasets (Modelacao de topicos e Analise de sentimento) sao iguais
-------------

In [18]:
df_topics = pd.read_pickle('Reviews_Lisboa_RJ_200k.pkl')
df_topics.head()

,date,comments,cidade,lang,char_count,word_count,texto_limpo,nomes_proprios,texto_anonimizado,texto_normalizado,texto_lematizado
0,2025-06-23,"Os anfitriões são maravilhosos, a casa é um charme com macaquinhos vindo toda manhã na varanda, porém considerando ter muitos móveis velhos e objetos pessoais acaba ficando com cheiro forte de coisa guardada e fechada",Rio de Janeiro,pt,217,35,Os anfitriões são maravilhosos a casa é um charme com macaquinhos vindo toda manhã na varanda porém considerando ter muitos móveis velhos e objetos pessoais acaba ficando com cheiro forte de coisa guardada e fechada,[],Os anfitriões são maravilhosos a casa é um charme com macaquinhos vindo toda manhã na varanda porém considerando ter muitos móveis velhos e objetos pessoais acaba ficando com cheiro forte de coisa guardada e fechada,os anfitriões são maravilhosos a casa é um charme com macaquinhos vindo toda manhã na varanda porém considerando ter muitos móveis velhos e objetos pessoais acaba ficando com cheiro forte de coisa guardada e fechada,anfitrião maravilhoso casa charme macaquinho manhã varanda considerar móvel velho objeto pessoal acabar ficar cheiro forte guardar fechar
1,2022-05-02,"A estadia com o Renato foi a melhor experiência no Airbnb até hoje .<br/>Vou falar 4 coisas que não poderia deixar passar.<br/>01- A vista da cobertura é sensacional , um ângulo de visão da Bahia de Guanabara , Pão de Açucar , Aeroporto Santos Dumont e Cristo Redentor é sensacional.<br/>02- A recepção no Chek-in foi indescritível , fomos recebidos com algumas surpresas que me deixaram de queixo caído , uma delas é de uma simpatia sem igual, mas não posso contar a vocês porque perderia a graça.<br/>03- A flexibilidade na reserva e as dicas do RJ que o Renato passa , são maravilhosas.<br/>04- A localização do apartamento é fantástica pois fica pórximo a tudo.<br/>Enfim , Super indico o Renato para se hospedarem , o Apartamento é fantastico !!!",Rio de Janeiro,pt,751,128,A estadia com o Renato foi a melhor experiência no Airbnb até hoje Vou falar coisas que não poderia deixar passar A vista da cobertura é sensacional um ângulo de visão da Bahia de Guanabara Pão de Açucar Aeroporto Santos Dumont e Cristo Redentor é sensacional A recepção no Chek in foi indescritível fomos recebidos com algumas surpresas que me deixaram de queixo caído uma delas é de uma simpatia sem igual mas não posso contar a vocês porque perderia a graça A flexibilidade na reserva e as dicas do RJ que o Renato passa são maravilhosas A localização do apartamento é fantástica pois fica pórximo a tudo Enfim Super indico o Renato para se hospedarem o Apartamento é fantastico,"[Renato, Renato, Renato]",A estadia com o host_anfitriao foi a melhor experiência no Airbnb até hoje Vou falar coisas que não poderia deixar passar A vista da cobertura é sensacional um ângulo de visão da Bahia de Guanabara Pão de Açucar Aeroporto Santos Dumont e Cristo Redentor é sensacional A recepção no Chek in foi indescritível fomos recebidos com algumas surpresas que me deixaram de queixo caído uma delas é de uma simpatia sem igual mas não posso contar a vocês porque perderia a graça A flexibilidade na reserva e as dicas do RJ que o host_anfitriao passa são maravilhosas A localização do apartamento é fantástica pois fica pórximo a tudo Enfim Super indico o host_anfitriao para se hospedarem o Apartamento é fantastico,a estadia com o host_anfitriao foi a melhor experiência no airbnb até hoje vou falar coisas que não poderia deixar passar a vista da cobertura é sensacional um ângulo de visão da bahia de guanabara pão de açucar aeroporto santos dumont e cristo redentor é sensacional a recepção no chek in foi indescritível fomos recebidos com algumas surpresas que me deixaram de queixo caído uma delas é de uma simpatia sem igual mas não posso contar a vocês porque perderia a graça a flexibilidade na reserva e as dicas do rj que o host_anfitriao passa são maravilhosas a localização do apartamento é fantástica pois fica pórximo

In [19]:
df_topics.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   date               200000 non-null  str   
 1   comments           200000 non-null  str   
 2   cidade             200000 non-null  str   
 3   lang               200000 non-null  str   
 4   char_count         200000 non-null  int64 
 5   word_count         200000 non-null  int64 
 6   texto_limpo        200000 non-null  str   
 7   nomes_proprios     200000 non-null  object
 8   texto_anonimizado  200000 non-null  str   
 9   texto_normalizado  200000 non-null  str   
 10  texto_lematizado   200000 non-null  str   
dtypes: int64(2), object(1), str(8)
memory usage: 16.8+ MB


In [20]:
df_sent = pd.read_pickle('Reviews_Lisboa_RJ_200k_texto_limpo.pkl')
df_sent.head()

,id,date,comments,cidade,lang,texto_limpo
0,1450109880518848792,2025-06-23,"Os anfitriões são maravilhosos, a casa é um charme com macaquinhos vindo toda manhã na varanda, porém considerando ter muitos móveis velhos e objetos pessoais acaba ficando com cheiro forte de coisa guardada e fechada",Rio de Janeiro,pt,"Os anfitriões são maravilhosos, a casa é um charme com macaquinhos vindo toda manhã na varanda, porém considerando ter muitos móveis velhos e objetos pessoais acaba ficando com cheiro forte de coisa guardada e fechada"
1,617993232260012422,2022-05-02,"A estadia com o Renato foi a melhor experiência no Airbnb até hoje .<br/>Vou falar 4 coisas que não poderia deixar passar.<br/>01- A vista da cobertura é sensacional , um ângulo de visão da Bahia de Guanabara , Pão de Açucar , Aeroporto Santos Dumont e Cristo Redentor é sensacional.<br/>02- A recepção no Chek-in foi indescritível , fomos recebidos com algumas surpresas que me deixaram de queixo caído , uma delas é de uma simpatia sem igual, mas não posso contar a vocês porque perderia a graça.<br/>03- A flexibilidade na reserva e as dicas do RJ que o Renato passa , são maravilhosas.<br/>04- A localização do apartamento é fantástica pois fica pórximo a tudo.<br/>Enfim , Super indico o Renato para se hospedarem , o Apartamento é fantastico !!!",Rio de Janeiro,pt,"A estadia com o Renato foi a melhor experiência no Airbnb até hoje . Vou falar 4 coisas que não poderia deixar passar. 01- A vista da cobertura é sensacional , um ângulo de visão da Bahia de Guanabara , Pão de Açucar , Aeroporto Santos Dumont e Cristo Redentor é sensacional. 02- A recepção no Chek-in foi indescritível , fomos recebidos com algumas surpresas que me deixaram de queixo caído , uma delas é de uma simpatia sem igual, mas não posso contar a vocês porque perderia a graça. 03- A flexibilidade na reserva e as dicas do RJ que o Renato passa , são maravilhosas. 04- A localização do apartamento é fantástica pois fica pórximo a tudo. Enfim , Super indico o Renato para se hospedarem , o Apartamento é fantastico ! ! !"
2,1490668744495176611,2025-08-18,"o apt do Maurício é otimo!!! tudo nele nos encantou. o prédio, os porteiros, o apt, a gentileza, a localização, enfim, um lugar com a cara do Rio e da gentileza e bom gosto do carioca",Rio de Janeiro,pt,"o apt do Maurício é otimo! ! ! tudo nele nos encantou. o prédio, os porteiros, o apt, a gentileza, a localização, enfim, um lugar com a cara do Rio e da gentileza e bom gosto do carioca"
3,1147883300514198966,2024-05-02,"muito organizado, bonito, limpo, funcional. anfitrião atencioso, os melhores horários de check-in e checkout que já vi. gelo, café, toalhas, roupa de cama tudo que precisei tinha. Supermercado e farmácia perto, excelente. a vila estava um pouco barulhenta a noite, mas entendo que isso está fora do controle do anfitrião. umica recomendação seria um insulfim espelhado na porta. mas pq eu sou mais sensível a claridade mesmo.",Rio de Janeiro,pt,"muito organizado, bonito, limpo, funcional. anfitrião atencioso, os melhores horários de check-in e checkout que já vi. gelo, café, toalhas, roupa de cama tudo que precisei tinha. Supermercado e farmácia perto, excelente. a vila estava um pouco barulhenta a noite, mas entendo que isso está fora do controle do anfitrião. umica recomendação seria um insulfim espelhado na porta. mas pq eu sou mais sensível a claridade mesmo."
4,1247881696472392123,2024-09-17,"Incrível! Amei tudo. Super rápida em soluções e respostas. Localização melhor, impossível!",Rio de Janeiro,pt,"Incrível! Amei tudo. Super rápida em soluções e respostas. Localização melhor, impossível!"


In [21]:
df_sent.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   id           200000 non-null  int64
 1   date         200000 non-null  str  
 2   comments     200000 non-null  str  
 3   cidade       200000 non-null  str  
 4   lang         200000 non-null  str  
 5   texto_limpo  200000 non-null  str  
dtypes: int64(1), str(5)
memory usage: 50.3 MB


In [22]:
# 1) Verificar se têm as mesmas colunas
mesmas_colunas = set(df_sent.columns) == set(df_topics.columns)

# 2) Contar o número de linhas
n_linhas_df_sent = df_sent.shape[0]
n_linhas_df_topics = df_topics.shape[0]

# 3) Comparar só as colunas em comum, ignorando o índice
colunas_comuns = ['comments', 'date', 'cidade']

mesmos_dados_comuns = (
    df_sent[colunas_comuns].reset_index(drop=True)
    .equals(df_topics[colunas_comuns].reset_index(drop=True))
)

print("Mesmas colunas:", mesmas_colunas)
print("Número de linhas no df_sent:", n_linhas_df_sent)
print("Número de linhas no df_topics:", n_linhas_df_topics)
print("Mesmos dados nas colunas comuns:", mesmos_dados_comuns)

Mesmas colunas: False
Número de linhas no df_sent: 200000
Número de linhas no df_topics: 200000
Mesmos dados nas colunas comuns: True
